[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/measure-a-skill/route_and_trace.ipynb)

# One skill is a file. Thirty is a context problem.

You have re-run our number on your own key against a prompt you wrote. So the
measurement works. This notebook is about what happens when you try to *use* the
thing you measured — and it starts by making your own approach fail in front of you.

**Runtime → Run all.**

| | needs | you get |
|---|---|---|
| Notebook 1 | nothing | the claim, the cases, the transcripts, and what we refuse to show |
| Notebook 2 | one free Google AI Studio key | the benchmark, re-run blind on your model |
| **You are here** | that same key, then a DecimalAI account | the token bill for doing it by hand, then routing, traces and activations |

The first cell block below runs on the model key alone. **Do not create an account
yet** — the ask comes after you have a reason, at cell 2.1.

Everything is pinned in
[`manifest.yaml`](https://github.com/decimal-labs/decimalai-python/blob/main/examples/measure-a-skill/manifest.yaml),
including which three skills get spliced. If you think the set was chosen to make a
point, swap it there and re-run: that is what pinning it in one file is for.

## Setup — the same manifest, one more key slot

Nothing to install yet. `decimalai` arrives at cell 2.2, when there is finally
something for it to do.

Keys are read from **Colab secrets** (🔑 in the left sidebar), never `input()`. A key
typed into a cell is a key in your browser history and in the `.ipynb` you later
share. This notebook wants `GOOGLE_API_KEY` now and `DECIMAL_API_KEY` at 2.1 — and
both are optional: with neither, every cell prints what it would have done and exits
cleanly.

In [ ]:
import json, os, re, subprocess, sys, textwrap, time
import requests, yaml

API = "https://api.decimal.ai/api/v1"
GENAI = "https://generativelanguage.googleapis.com/v1beta"
MANIFEST_URL = "https://raw.githubusercontent.com/decimal-labs/decimalai-python/main/examples/measure-a-skill/manifest.yaml"


def get(path, timeout=30, **params):
    """GET with backoff. Distinguishes 'rate limited' from 'actually missing'.

    Retries TRANSPORT failures too, not just retryable status codes. Tier 0 only
    retried on status, and this notebook pulls the ~50KB per-skill benchmark payload
    — the one call slow enough to hit a read timeout on a warm-starting backend. A
    ReadTimeout there used to propagate straight out of the cell as a traceback.
    """
    url = path if path.startswith("http") else f"{API}{path}"
    delay, last = 1.0, ""
    for attempt in range(5):
        try:
            r = requests.get(url, params=params or None, timeout=timeout)
        except requests.RequestException as exc:
            last = type(exc).__name__
            time.sleep(delay)
            delay *= 2
            continue
        if r.status_code == 200:
            return r.json() if "json" in r.headers.get("content-type", "") else r.text
        if r.status_code in (429, 500, 502, 503, 504):
            last = str(r.status_code)
            time.sleep(delay)
            delay *= 2
            continue
        raise RuntimeError(f"{r.status_code} on {url}")
    raise RuntimeError(f"gave up after 5 attempts ({last}) on {url}")


def wrap(text, width=88, indent="  "):
    return textwrap.indent(textwrap.fill(str(text), width), indent)


def est(text):
    """chars/4 — the same cheap estimate the platform's routing budget uses.

    Within ~20% on English markdown, needs no tokenizer and no key. Every token
    figure printed with a `~` came from here; every figure without one came from
    the model's own `usageMetadata`, which is what actually gets billed.
    """
    return (len(text) + 3) // 4


def secret(names):
    """First hit among Colab secrets, then the environment. Returns (value, where).

    `userdata.get` raises for 'no such secret' AND for 'notebook access not granted'
    — two different problems with the same one-click fix — so both are swallowed and
    the caller prints one instruction instead of a stack trace.
    """
    try:
        from google.colab import userdata  # noqa: F401
        for n in names:
            try:
                v = (userdata.get(n) or "").strip()
                if v:
                    return v, f"Colab secret {n}"
            except Exception:
                pass
    except ImportError:
        pass
    for n in names:
        v = (os.environ.get(n) or "").strip()
        if v:
            return v, f"env {n}"
    return "", ""


# Same discipline as Tier 0: the manifest lives on `main` so a stale figure is a
# one-line YAML edit. The `tier2` block may legitimately be ABSENT — a tab opened
# against an older `main` must get these values and a note, not a KeyError.
FALLBACK = {
    "demo_skill": {"slug": "flsa-exemption-test", "version": 1, "demo_case": "case-01"},
    "tier2": {
        "splice_skills": [
            {"slug": "flsa-exemption-test", "version": 1,
             "why": "matches the question"},
            {"slug": "progressive-tax-bracket-math", "version": 1,
             "why": "matches the WORDS in the question ($80/hour, 40 hours) and nothing else"},
            {"slug": "markdown-semantic-wrap", "version": 1,
             "why": "matches nothing, and still imposes a house output format on every answer"},
        ],
        "lane_cases": 8, "route_top_k": 1, "agent_name": "notebook-tier2",
        "model_candidates": ["gemini-3.6-flash", "gemini-3.5-flash",
                             "gemini-2.5-flash", "gemini-2.0-flash"],
        "model_key_secrets": ["GOOGLE_API_KEY", "GEMINI_API_KEY"],
        "decimal_key_secrets": ["DECIMAL_API_KEY", "DECIMALAI_API_KEY"],
        "routing_desc_token_budget": 1500, "routing_max_menu_rows": 30,
    },
}

try:
    M = yaml.safe_load(requests.get(MANIFEST_URL, timeout=10).text)
    if not isinstance(M, dict) or "demo_skill" not in M:
        raise ValueError("manifest did not parse as expected")
    print("manifest: fetched from main")
except Exception as exc:
    M = FALLBACK
    print(f"manifest: using the embedded copy ({type(exc).__name__}) — "
          "figures may lag what is on main")

DEMO = M["demo_skill"]
T2 = M.get("tier2")
if not T2:
    T2 = FALLBACK["tier2"]
    print("manifest: no `tier2:` block on main yet — using the embedded staging")

MODEL_KEY, MODEL_KEY_FROM = secret(T2["model_key_secrets"])

# Ask the key what it can reach instead of pinning a model id. A hard-pinned id is
# the single most reliable way to make a notebook 404 in six months, and the free
# AI Studio tier rotates what it serves.
MODEL, WHY_NO_MODEL = "", "no model key"
if MODEL_KEY:
    WHY_NO_MODEL = ""
    try:
        r = requests.get(f"{GENAI}/models",
                         params={"key": MODEL_KEY, "pageSize": 200}, timeout=30)
        if r.status_code == 200:
            live = {m["name"].split("/")[-1] for m in (r.json().get("models") or [])
                    if "generateContent" in (m.get("supportedGenerationMethods") or [])}
            MODEL = next((m for m in T2["model_candidates"] if m in live), "")
            if not MODEL:
                WHY_NO_MODEL = "this key reaches none of the manifest's candidate models"
        else:
            WHY_NO_MODEL = f"the model key was rejected (HTTP {r.status_code})"
    except Exception as exc:
        WHY_NO_MODEL = f"could not reach the model API ({type(exc).__name__})"

print(f"model key:  {MODEL_KEY_FROM or '— none —'}")
print(f"model:      {MODEL or '— none — ' + WHY_NO_MODEL}")
if MODEL and MODEL != T2["model_candidates"][0]:
    print(wrap(f"Note: the registry number was measured on {T2['model_candidates'][0]}, "
               f"which this key cannot reach. {MODEL} is a different model, and lift is "
               "model-relative — so treat every accuracy figure below as a statement "
               f"about {MODEL}, not about the skill in general."))


def ask(system, user):
    """One generateContent call → (text, prompt_tokens, finish_reason).

    Raw REST on purpose: nothing to pip install yet, and
    `usageMetadata.promptTokenCount` is the SERVER's count of what actually reached
    the model. That is the number this entire notebook is arguing about, so it should
    not be an estimate.

    No `maxOutputTokens`, deliberately. These models think before they answer and the
    thinking spends the output budget; a cap low enough to feel thrifty returns an
    empty `parts` with finishReason MAX_TOKENS, which grades as a WRONG ANSWER — and
    would quietly corrupt every arm below by the same invisible amount.
    """
    if not (MODEL and MODEL_KEY):
        return None, 0, "no-model"
    payload = {"contents": [{"role": "user", "parts": [{"text": user}]}],
               "generationConfig": {"temperature": 0}}
    if system:
        payload["systemInstruction"] = {"parts": [{"text": system}]}
    delay = 2.0
    for attempt in range(4):
        r = requests.post(f"{GENAI}/models/{MODEL}:generateContent",
                          params={"key": MODEL_KEY}, json=payload, timeout=120)
        if r.status_code == 200:
            d = r.json()
            cand = (d.get("candidates") or [{}])[0]
            parts = (cand.get("content") or {}).get("parts") or []
            text = "".join(p.get("text", "") for p in parts if isinstance(p, dict))
            ptok = int((d.get("usageMetadata") or {}).get("promptTokenCount") or 0)
            return text, ptok, cand.get("finishReason") or "STOP"
        if r.status_code in (429, 500, 502, 503):
            time.sleep(delay)
            delay *= 2
            continue
        return None, 0, f"HTTP {r.status_code}"
    return None, 0, "rate-limited"

## 2.0 — The thing you are about to do by hand

You liked one skill enough to verify it. The obvious next move is to keep the ones
you like in a folder and paste them all into your system prompt. That is what almost
everyone does, it is genuinely the right call at one skill, and it has a bill.

Below: three real registry skills, spliced into one system prompt, and the **same
lane of cases** from the run behind our headline number. Two arms, one variable —
which bodies are in the prompt.

The three are pinned in the manifest and each fails differently: one matches the
question, one matches the *words* in the question (an `$80/hour, 40 hours a week`
fact pattern reads a lot like tax math), and one matches nothing at all but carries a
house output-format contract that says *apply whenever you write markdown*. That
third one is the whole problem. It has no opinion about employment law and it still
gets a vote on every answer.

The relevant skill is spliced **first**, which is the friendliest possible
arrangement for the arm being criticised. No DecimalAI account is involved in this
cell — every byte below is fetched anonymously.

Grading is six lines of Python, not a judge: this suite states every expectation as
*"the JSON output sets the `exempt` field to NO"*, which is machine-checkable. Before
using it the cell **calibrates that grader against the registry's own judge** on the
answers the real run recorded, and prints the disagreement. It is not zero, and the
reason is interesting.

In [ ]:
# ── the skills folder you have not built yet, built for you ─────────
def fetch_body(slug, version):
    """A pinned SKILL.md, or '' with a printed reason. Public, anonymous, no key."""
    try:
        return get(f"https://app.decimal.ai/s/{slug}@{version}/SKILL.md")
    except Exception as exc:
        print(f"  could not fetch {slug}@{version} ({type(exc).__name__}) — "
              "dropped from this run; re-run the cell")
        return ""


WHY = {e["slug"]: e["why"] for e in T2["splice_skills"]}
BODIES, SPLICE = {}, []
for entry in T2["splice_skills"]:
    body = fetch_body(entry["slug"], entry["version"])
    if body:
        BODIES[entry["slug"]] = body
        SPLICE.append(entry["slug"])

TARGET = DEMO["slug"]
if TARGET not in BODIES:
    # The manifest's gates fell through to the warm spare, so the pinned splice set no
    # longer contains the skill being measured. Put it back, first, or the two arms
    # would differ by more than one variable.
    body = fetch_body(TARGET, DEMO["version"])
    if body:
        BODIES[TARGET] = body
        SPLICE.insert(0, TARGET)
        WHY.setdefault(TARGET, "matches the question")

print("your skills folder — files fetched anonymously, no account:\n")
for slug in SPLICE:
    print(f"  {len(BODIES[slug]):>6,} bytes  ~{est(BODIES[slug]):>5,} tok  {slug}")
    print(f"                                {WHY.get(slug, '')}")


def splice(slugs):
    return "\n\n---\n\n".join(BODIES[s] for s in slugs if s in BODIES)


ONE, ALL = splice([TARGET]), splice(SPLICE)
print(f"\n  one skill   ~{est(ONE):>6,} tok of system prompt")
print(f"  all {len(SPLICE)}       ~{est(ALL):>6,} tok  "
      f"({est(ALL) / max(est(ONE), 1):.1f}x), on every request, forever")

# ── the lane: real cases from the run behind the registry number ────
# 'The JSON output sets the exempt field to NO.' — the FLSA suite states every
# expectation as a field assertion, so this lane can be graded in six lines of
# Python with no judge at all. That is NOT true of every suite in the registry (the
# warm spare's expectations are prose), so a case that will not parse is DROPPED
# rather than guessed at, and the drop count is printed.
FIELD = re.compile(r"sets the `?(\w+)`? field to `?([^`.]+?)`?\.?\s*$", re.I)


def checks(case):
    exps = (case.get("expectation_results")
            or case.get("without_expectation_results") or [])
    want = {}
    for e in exps:
        m = FIELD.search((e.get("expectation") or "").strip())
        if not m:
            return None
        want[m.group(1)] = m.group(2).strip()
    return want or None


# The heaviest call in the notebook (~50KB) and the only one that fails often enough
# to matter. If it does, the accuracy half of this cell is unavailable and the token
# half — which is arithmetic on bytes already in hand — carries on regardless.
cases, lane = [], []
try:
    run = get(f"/registry/skills/{TARGET}/benchmark", timeout=90)["latest_run"]
    cases = sorted(run["results"], key=lambda r: r["case_name"])
except Exception as exc:
    print(f"\nlane: unavailable — could not fetch the benchmark run "
          f"({type(exc).__name__}). The token figures above still stand.")

if cases:
    lane = [c for c in cases if c.get("case_prompt") and checks(c)][: T2["lane_cases"]]
    withheld = sum(1 for c in cases if not c.get("case_prompt"))
    unparsed = sum(1 for c in cases if c.get("case_prompt") and not checks(c))
    print(f"\nlane: {len(lane)} cases (of {len(cases)} in the suite; "
          f"{withheld} prompt-withheld, {unparsed} not machine-checkable)")
    print(wrap("It is the HEAD of the suite in case order, not a hand-picked set. "
               "Choosing the cases would be choosing the answer."))

In [ ]:
def answer_json(text):
    """The first {...} in the reply, fences and commentary stripped."""
    if not text:
        return None
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def grade(text, want):
    """Exact match on every asserted field. Deliberately stricter than the registry's
    judge, which is an LLM reading the same sentences and will accept a partial.
    """
    obj = answer_json(text)
    if obj is None:
        return False, "no JSON object in the reply"
    for field, expected in want.items():
        got = str(obj.get(field, "")).strip().lower()
        if got != expected.strip().lower():
            return False, f"{field}={obj.get(field)!r}, expected {expected!r}"
    return True, ""


# ── calibrate this grader against theirs, BEFORE trusting it ────────
# Every case in the payload carries the answers the registry's own run recorded and
# the verdict its LLM judge gave. Re-grading those recorded answers with the function
# above costs nothing and says exactly how far this grader sits from the one behind
# the headline. Running an instrument you have not checked is how you end up arguing
# about a number that was never measuring what you thought.
PASSING = {"flip_to_pass", "pass_kept"}
checkable = [c for c in cases if checks(c)]
agree = sum(1 for c in checkable
            if grade(c.get("with_skill_output"), checks(c))[0]
            == (c["outcome"] in PASSING))
if checkable:
    print(f"grader calibration: agrees with the registry's judge on {agree}/"
          f"{len(checkable)} recorded with-skill answers\n")
    if agree < len(checkable):
        print(wrap(
            f"The {len(checkable) - agree} disagreements are worth understanding "
            "before you read anything below. Two causes, both real. (1) This grader "
            "does not accept a partial: where the judge scored a verbose answer 0.33 "
            "and called it a pass, exact match calls it a miss. (2) The suite drifted "
            "— several cases carry a recorded FAIL alongside expectation text that "
            "today's answer satisfies, which is the in-place eval rewrite notebook 1 "
            "quantified, seen from the other side."))
        print()
        print(wrap(
            "So: the percentages below are NOT a re-derivation of the +pts headline, "
            "and comparing them to it would be comparing two different graders on "
            "different case sets. That is notebook 2's job. What this lane needs is "
            "weaker and sufficient — one grader, held fixed, across three arms that "
            "differ in exactly one thing."))
        print()

RESULTS = {}


def run_lane(label, prepare):
    """Run every lane case through one arm.

    `prepare(case)` returns (system_prompt, extra_model_calls, extra_prompt_tokens,
    note) so an arm that needs a round-trip of its own — the routed one, at 2.3 —
    charges itself for it instead of hiding it in the margin.
    """
    if not (MODEL and lane and ONE):
        why = (WHY_NO_MODEL or ("the skill under test could not be fetched"
                                if not ONE else "no gradeable cases in this suite"))
        print(f"  {label:<20} skipped — {why}")
        return None
    ok, ptok, calls, broke, picks = 0, 0, 0, [], []
    for c in lane:
        system, pre_calls, pre_ptok, note = prepare(c)
        text, pt, finish = ask(system, c["case_prompt"])
        calls += 1 + pre_calls
        ptok += pt + pre_ptok
        passed, why = grade(text, checks(c))
        ok += 1 if passed else 0
        if not passed:
            broke.append((c["case_name"], why or f"finish={finish}"))
        if note:
            picks.append(note)
    r = {"label": label, "ok": ok, "n": len(lane), "broke": broke, "picks": picks,
         "ptok": ptok / len(lane), "calls": calls / len(lane)}
    RESULTS[label] = r
    print(f"  {label:<20} {ok}/{len(lane)} correct   "
          f"{r['ptok']:>7,.0f} prompt tok/req   {r['calls']:.1f} model call/req")
    return r


print("two arms, one variable — which bodies are in the prompt:\n")
arm_one = run_lane("single skill", lambda c: (ONE, 0, 0, ""))
arm_all = run_lane("all three spliced", lambda c: (ALL, 0, 0, ""))

print()
if arm_one and arm_all:
    mult = arm_all["ptok"] / max(arm_one["ptok"], 1)
    marginal = (est(ALL) - est(ONE)) / max(len(SPLICE) - 1, 1)
    print(wrap(
        f"Every request now carries {mult:.1f}x the prompt tokens — measured by the "
        "model, not estimated — whether or not the extra skills had anything to say. "
        f"Each further file in that folder costs about {marginal:,.0f} more tokens on "
        "EVERY request; ten of them is roughly "
        f"{est(ONE) + 9 * marginal:,.0f} tokens before your user has typed anything."))
    print()
    new_breaks = [(n, w) for n, w in arm_all["broke"]
                  if n not in {x for x, _ in arm_one["broke"]}]
    if arm_all["ok"] < arm_one["ok"]:
        print(wrap(f"And it cost accuracy: {arm_one['ok']}/{arm_one['n']} -> "
                   f"{arm_all['ok']}/{arm_all['n']}. What broke:"))
        for name, why in new_breaks:
            print(f"      {name}: {why}")
        print()
        print(wrap("Nothing here told the model to apply a markdown wrapping rule or a "
                   "tax-bracket checklist to an overtime question. Splicing did. A "
                   "skill in the prompt is a skill with a vote, and you are paying for "
                   "the vote on every request."))
    elif arm_all["ok"] == arm_one["ok"]:
        print(wrap("On this run the extra skills changed no answers. The honest reading "
                   "is that you paid the token bill and got nothing for it — not that "
                   "splicing is free. Eight cases at temperature 0 is one sample; the "
                   "token column is arithmetic and does not need a sample."))
    else:
        print(wrap(f"On this run the extra skills HELPED "
                   f"({arm_one['ok']}/{arm_one['n']} -> {arm_all['ok']}/{arm_all['n']}). "
                   "We print it either way — a demo that can only produce one outcome "
                   "is not a measurement. The token bill is unchanged, so the argument "
                   "from here is about cost rather than correctness."))
else:
    print(wrap(f"Only half of this cell ran, and it is the half that is arithmetic: "
               f"~{est(ALL):,} tokens of skill go into every request under "
               f"splice-everything, against ~{est(ONE):,} for the one skill you "
               f"measured. That ratio needs no model to be true. Add a Google AI "
               "Studio key (notebook 2 shows how) and re-run to see what it does to "
               "the answers."))

## 2.1 — Now the account is worth something

Splicing every skill into every request multiplied your prompt bill and handed an
irrelevant policy a vote on an employment-law question. The fix is not a bigger
context window — it is **deciding what goes in, per request**. That decision needs a
server that knows which skills you have and how they have performed, which means an
account.

**What signing up actually does**, so nothing below is a surprise:

- A `dai_sk_` key is minted for you at signup. You do not have to detour through
  Settings before `decimalai.init(api_key=...)` works — though that is also where you
  can find it again, revoke it, or mint another.
- Your workspace starts **genuinely empty**, on purpose — a demo seed exists and you
  can choose it, but nothing is written for you. Cell 2.2 is what puts your first
  real thing in the workspace.
- Routing (`smart_route`) is on **every plan, including Free**. Our own SDK docstring
  and API reference both said "Pro tier" until recently. They were wrong about our
  own pricing, in the direction that discourages exactly the people this registry is
  for.

Sign up at **[app.decimal.ai](https://app.decimal.ai)**, then add the key as a Colab
secret named `DECIMAL_API_KEY` (🔑 in the left sidebar → *Add new secret* → toggle
*Notebook access*) and re-run this cell. Skipping is fine — everything below prints
what it would have done.

In [ ]:
DAI_KEY, DAI_KEY_FROM = secret(T2["decimal_key_secrets"])

arm_one, arm_all = RESULTS.get("single skill"), RESULTS.get("all three spliced")
if arm_one and arm_all:
    broke_new = [n for n, _ in arm_all["broke"]
                 if n not in {x for x, _ in arm_one["broke"]}]
    print(wrap(f"Splicing every skill into every request cost you "
               f"{arm_all['ptok'] / max(arm_one['ptok'], 1):.1f}x the prompt tokens "
               f"and broke {len(broke_new)} case(s) the single skill got right. "
               "Routing is the fix, and routing needs an account.", 88, ""))
else:
    print("(2.0 could not run its arms, so there is no measured pain to quote here.)")
print()

if DAI_KEY:
    print(f"DecimalAI key: {DAI_KEY_FROM}  ({DAI_KEY[:7]}…)")
    if not DAI_KEY.startswith("dai_sk_"):
        print(wrap("That does not look like a DecimalAI key — ours start `dai_sk_`. "
                   "If you pasted your model key into this slot, the cells below will "
                   "401 and say so rather than pretending."))
else:
    print("DecimalAI key: — none — everything from here prints a skip notice.")
    print()
    print(wrap("To continue: app.decimal.ai → sign up → the key is minted for you "
               "(Settings → API keys if you need it again) → Colab 🔑 → Add new "
               "secret named DECIMAL_API_KEY → toggle Notebook access → re-run from "
               "this cell."))

## 2.2 — Put the skills where the Router can see them

This is the step everyone forgets, and it fails silently when you do.

**The Skill Router serves what your workspace owns or links — never the public
registry.** Route against an empty workspace and you get a valid `200`, an empty
skill list, an empty prompt fragment, and a demo that appears to run while doing
nothing at all. So this cell installs first and then *checks*, loudly, before
anything downstream is allowed to look successful.

`decimalai skills install` does two things: forks the registry skill into your
workspace (tracked and versioned, so a new upstream version is a diff you choose) and
writes `SKILL.md` to disk for agent runtimes. The two halves are separate commands
now — `skills export` writes the files without a copy, and the Install button /
`router.use()` links without one — but `install` is the one that does the thing this
cell needs. If you only want the file and no account at all, that is `decimalai
skills pull`, which is anonymous.

In [ ]:
ROUTING_OK, INSTALLED = False, []


def sh(args, env_extra=None):
    """Run a command, return (exit_code, combined_output). Never raises."""
    env = dict(os.environ)
    env.update(env_extra or {})
    p = subprocess.run(args, capture_output=True, text=True, env=env)
    return p.returncode, (p.stdout + p.stderr).strip()


# `!pip install -q decimalai` is the notebook idiom, but it is not Python — a file
# that uses it cannot be ast-parsed, linted or run as a script, and this notebook is
# checked by all three. Same effect, and we get to read the exit code. The install is
# unconditional: it needs no key, and cell 2.4 imports the package whether or not you
# ever signed up.
rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "decimalai"])
print(f"pip install decimalai … {'ok' if rc == 0 else 'FAILED'}")
if rc:
    print(textwrap.indent(out[-600:], "      "))

if not DAI_KEY:
    print("\nfork + menu check skipped — no DecimalAI key.")
    print(wrap("What the rest of this cell would do: "
               f"`decimalai skills install` for {', '.join(SPLICE)}, then confirm the "
               "Router's menu is non-empty before anything downstream is allowed to "
               "look successful."))
else:
    for slug in SPLICE:
        rc, out = sh([sys.executable, "-m", "decimalai.cli", "skills", "install", slug],
                     {"DECIMAL_API_KEY": DAI_KEY})
        print(f"  {slug:<32} {'forked into your workspace' if rc == 0 else 'FAILED'}")
        if rc:
            print(textwrap.indent(out[-400:], "      "))
        else:
            INSTALLED.append(slug)

    # THE CHECK. Asked over raw HTTP rather than SkillRouter.get_menu() on purpose:
    # get_menu degrades gracefully and returns {"skills": []} for a 401 exactly as it
    # does for an empty workspace. Two very different problems, one identical symptom,
    # and the remediation below has to name the right one. (A bare `assert` would be
    # shorter and would end this notebook in a traceback, which teaches nothing.)
    menu, status = {}, None
    try:
        r = requests.get(f"{API}/skills/menu",
                         headers={"Authorization": f"Bearer {DAI_KEY}"}, timeout=30)
        status = r.status_code
        menu = r.json() if status == 200 else {}
    except Exception as exc:
        status = f"transport: {type(exc).__name__}"
    names = [s.get("name") for s in (menu.get("skills") or []) if s.get("name")]

    print()
    if status in (401, 403):
        print("  ✗ THE ROUTER REFUSED YOUR KEY — nothing below can work.")
        print(wrap(f"HTTP {status} from GET /skills/menu. The key is wrong, revoked, or "
                   "belongs to another workspace. Mint a fresh one at "
                   "app.decimal.ai → Settings → API keys, replace the DECIMAL_API_KEY "
                   "Colab secret, and re-run from 2.1."))
    elif status != 200:
        print(f"  ✗ COULD NOT READ THE MENU ({status}) — stopping rather than "
              "pretending.")
        print(wrap("This is a platform or network problem, not a mistake you made. "
                   "Re-run the cell; if it persists, email hello@decimal.ai."))
    elif not names:
        print("  ✗ THE ROUTER'S MENU IS EMPTY — every cell below would run and do "
              "nothing.")
        print(wrap(f"Your key is valid and your workspace owns no skills, so routing "
                   f"has nothing to route. {len(INSTALLED)} of {len(SPLICE)} installs "
                   f"reported success above. Either they failed (their output is "
                   f"printed) or they forked into a different workspace than this key "
                   f"reads. Fix: run `decimalai skills install {SPLICE[0]}` in a "
                   f"terminal with the same DECIMAL_API_KEY, confirm it appears at "
                   f"app.decimal.ai/skills, then re-run this cell."))
    else:
        ROUTING_OK = True
        print(f"  ✓ menu: {len(names)} skill(s) the Router can serve you")
        for n in names[:12]:
            print(f"      {n}")

## 2.3 — Route instead of splice

Same lane, same grader, same model. The only change is that a server decides which
bodies enter the prompt, per request, instead of all of them entering every request.

`POST /api/v1/skills/route` is free on every plan, including Free — worth stating
plainly, because the SDK docstring and API reference described it as Pro-tier until
recently and that was incorrect.

One thing the response does **not** contain: a skill body. Routing returns menu rows —
name, description, and a relevance score when it ranked. Turning a row into text the
model can act on is a second fetch, which the SDK does for you behind
`inject_skill_body=True`; here it is spelled out so you can see what is being paid
for.

Read the `strategy` line in the output before the table. It changes what the table
means.

In [ ]:
ROUTING_ID, OFFERED = None, []
H = {"Authorization": f"Bearer {DAI_KEY}", "Content-Type": "application/json"}


def route(query):
    """POST /skills/route. Returns the decision, or {} on any failure."""
    try:
        r = requests.post(f"{API}/skills/route", headers=H, timeout=60,
                          json={"query": query, "top_k": T2["route_top_k"],
                                "include_attachments": False})
        return r.json() if r.status_code == 200 else {}
    except Exception:
        return {}


def body_of(name):
    """The workspace copy of a routed skill's body. The SDK one-liner for this is
    SkillRouter(api_key=...).get_skill_body(name); raw HTTP here to keep this cell
    readable as a protocol rather than as an SDK tour.
    """
    try:
        r = requests.get(f"{API}/skills/{name}/body", headers=H, timeout=30)
        return (r.json().get("body") or "") if r.status_code == 200 else ""
    except Exception:
        return ""


ROUTED_READY = False
if not (ROUTING_OK and lane):
    print("skipped — routing needs a key and a non-empty workspace menu (see 2.2).")
else:
    probe = route(lane[0]["case_prompt"])
    ROUTING_ID = probe.get("routing_id")
    OFFERED = [s.get("name") for s in (probe.get("skills") or []) if s.get("name")]
    fragment = probe.get("prompt_fragment") or ""
    ROUTED_READY = bool(OFFERED)
    print(f"  strategy    {probe.get('strategy') or '—'}")
    print(f"  offered     {', '.join(OFFERED) or '—'}")
    print(f"  routing_id  {ROUTING_ID}")
    print(f"  fragment    {len(fragment):,} chars (~{est(fragment):,} tok) — "
          "menu rows, no bodies")
    print()
    if not ROUTED_READY:
        # 2.2 proved the menu is non-empty, so an empty routing decision here is the
        # ROUTE endpoint failing, not an empty workspace — a different problem with
        # the same shape, again.
        print(wrap("POST /skills/route returned nothing, even though 2.2 confirmed your "
                   "menu is not empty. That is the routing endpoint failing rather than "
                   "an empty workspace — re-run the cell, and if it persists check "
                   "app.decimal.ai/settings for plan status. The routed arm below will "
                   "skip rather than report a zero it cannot explain."))
    elif probe.get("strategy") == "full_menu":
        print(wrap(
            f"`full_menu` is not a failure — it is the router being honest about your "
            f"workspace. Your whole menu costs {probe.get('desc_tokens', est(fragment))} "
            f"of its {T2['routing_desc_token_budget']}-token description budget across "
            f"{probe.get('rows_total', len(OFFERED))} skills, so it returned all of them "
            "and skipped the embedding call entirely. Selection binds once the menu "
            f"stops fitting — past {T2['routing_desc_token_budget']:,} description "
            f"tokens or {T2['routing_max_menu_rows']} skills."))
        print()
        print(wrap(
            "So there is no ranking to read here, and the tempting shortcut — splice "
            "'the top result' — would be splicing whatever sorts first by NAME. Ours "
            "happens to be the right one alphabetically, so that table would look like "
            "routing worked when it was luck. Instead the arm below selects the way the "
            "SDK ships it: the menu goes to the model, one name comes back, and only "
            "that body is paid for. That is the load_skill loop, and it works at three "
            "skills and at three hundred."))
    print()

In [ ]:
def routed_prepare(case):
    """Route one request, then splice back only what routing chose."""
    d = route(case["case_prompt"])
    names = [s.get("name") for s in (d.get("skills") or []) if s.get("name")]
    scored = bool(d.get("skills")) and all(
        "relevance" in (s or {}) for s in d["skills"])
    pre_calls, pre_ptok = 0, 0
    if scored:
        picked = names[: T2["route_top_k"]]
    else:
        reply, pre_ptok, _ = ask(
            (d.get("prompt_fragment") or "")
            + "\n\nName the ONE skill above that applies to the request that follows. "
              "Answer with its name and nothing else.",
            case["case_prompt"])
        pre_calls = 1
        want = (reply or "").strip().strip("`*.\"' ").lower()
        picked = ([n for n in names if n.lower() == want]
                  or [n for n in names if want and n.lower() in want])[: T2["route_top_k"]]
    bodies = [b for b in (body_of(n) or BODIES.get(n, "") for n in picked) if b]
    return "\n\n---\n\n".join(bodies), pre_calls, pre_ptok, ",".join(picked) or "—"


if not (ROUTING_OK and ROUTED_READY and lane and MODEL):
    print("skipped — the routed arm needs a DecimalAI key, a live routing decision "
          "and a model key.")
else:
    arm_routed = run_lane("routed", routed_prepare)
    print()
    print("  arm                  prompt tok/req   model calls/req   correct")
    for key in ("all three spliced", "routed", "single skill"):
        r = RESULTS.get(key)
        if r:
            print(f"  {key:<20} {r['ptok']:>13,.0f}   {r['calls']:>15.1f}   "
                  f"{r['ok']}/{r['n']}")
    if arm_routed and arm_routed["picks"]:
        chosen = {}
        for p in arm_routed["picks"]:
            chosen[p] = chosen.get(p, 0) + 1
        print("\n  what routing chose, per request: "
              + ", ".join(f"{k} x{v}" for k, v in sorted(chosen.items())))

    spliced = RESULTS.get("all three spliced")
    single = RESULTS.get("single skill")
    print()
    if arm_routed and spliced:
        overhead = ("one platform round-trip per request, plus the extra model call "
                    "the menu selection needed"
                    if arm_routed["calls"] > 1.05 else
                    "one platform round-trip per request, and no extra model call — "
                    "the server had already ranked, so there was nothing to ask")
        print(wrap(
            f"Routing cost {arm_routed['ptok']:,.0f} prompt tokens per request against "
            f"{spliced['ptok']:,.0f} for splicing everything. What it paid to get there: "
            f"{overhead}. Both are in the table; neither is hidden in the margin."))
    if arm_routed and single:
        print()
        print(wrap(
            f"The bottom row is the ceiling: one skill, chosen by hand, by someone who "
            f"already knew the answer. Routing scored {arm_routed['ok']}/{arm_routed['n']} "
            f"against that {single['ok']}/{single['n']} without being told which skill "
            "the question was about. The gap between those two columns is the honest "
            "measure of what routing costs you in accuracy for what it saves you in "
            "context — and if that gap is zero on your run, say so out loud, because "
            "eight cases is eight cases."))

## 2.4 — Trace it, with or without a key

Routing decided; now record what happened, so next month's version of this question
is answered from your traffic instead of from our benchmark.

The block below has **no `if key:` branch around the tracing calls**.
`decimalai.init(enabled=False)` is a real no-op — no client is constructed, no
background sender starts, and the trace's `_send()` returns before it touches the
network — so the identical code path runs whether or not you have an account. That is
worth more than it sounds: the alternative is two code paths, one of which is only
exercised by people who are not paying you, which is the path that rots.

Don't take that on faith. The cell prints `export_status()` at the end: `sent=0,
failed=0` with no key is the claim, checked in front of you.

Two things this cell deliberately does not do:

- **`init(google=True)`** — the auto-instrumentor for the Google GenAI SDK ships only
  in this package's *test* extras, so on a plain `pip install decimalai` it logs a
  pip hint and traces nothing. It looks like it worked. (`openai=True` has the same
  shape: the `openai` extra installs the provider SDK, not the instrumentor.)
- **`init(enabled=False, langchain=True)`** — never combine those. The framework
  flags are applied *outside* the `enabled` gate, and the LangChain install submits a
  background skill pull that calls the API and writes `SKILL.md` files to disk, with
  tracing nominally disabled.

In [ ]:
# The one branch that is NOT about your key: if 2.2's pip step could not run there is
# no package to demonstrate. Everything after this point is key-blind.
try:
    import decimalai
except ImportError:
    decimalai = None
    print("skipped — `decimalai` is not importable; re-run 2.2 and check its pip line.")

# The only conditional inside the trace block is the model call itself: with no model
# key there is no LLM call to record, and inventing one would put a fiction in your
# trace store. Everything DecimalAI-side runs identically either way.
case = lane[0] if lane else None
text, ptok, latency_ms = None, 0, 0
if MODEL and case:
    t0 = time.time()
    text, ptok, _finish = ask(ONE, case["case_prompt"])
    latency_ms = int((time.time() - t0) * 1000)

if decimalai is not None:
    # verify=True (the default) probes the backend during init and raises on 401/403 —
    # failing loud beats a silent day of background 401s. Caught here because this
    # notebook's contract is 'no traceback', not because you should catch it in yours.
    try:
        decimalai.init(api_key=DAI_KEY or None, enabled=bool(DAI_KEY))
        print(f"decimalai {decimalai.__version__} — tracing "
              f"{'ENABLED' if DAI_KEY else 'disabled (no-op mode)'}")
    except Exception as exc:
        print(f"init refused the key ({type(exc).__name__}: {str(exc)[:120]}) — "
              "continuing; the calls below stay no-ops")

    with decimalai.start_trace(agent_name=T2["agent_name"],
                               session_id="measure-a-skill-tier2") as trace:
        if case:
            trace.set_input(case["case_prompt"])
        # The routing decision from 2.3. This is what closes the join between what was
        # offered and what actually got used — without it the two halves are unrelated
        # rows in two tables.
        trace.set_routing_id(ROUTING_ID)
        trace.log_skill_offered(names=OFFERED or [TARGET])
        trace.log_tool_call(name="load_skill", input=TARGET,
                            output=f"{len(ONE):,} chars of SKILL.md")
        if text is not None and case:
            trace.log_llm_call(model=MODEL, provider="google",
                               input=[{"role": "user",
                                       "content": case["case_prompt"]}],
                               output={"text": text}, input_tokens=ptok,
                               latency_ms=latency_ms, finish_reason="stop",
                               call_role="final")
        # offered → loaded → ACTIVATED. Only you can assert the last one: it means the
        # skill changed the answer, which no amount of prompt inspection can prove.
        trace.log_skill_activation(name=TARGET)
        if text:
            trace.set_output(text)

    decimalai.flush()
    st = decimalai.export_status()
    print(f"  sent {st.sent}   failed {st.failed}   "
          f"last error: {st.last_error or 'none'}")
    if DAI_KEY and st.sent == 0 and st.failed == 0:
        # flush() waits 5s; a rate-limited send retries past that, so 0/0 WITH a key
        # means 'still in flight', not 'nothing happened'. The SDK prints its own
        # one-line summary at interpreter exit — that is the authoritative read.
        print(wrap("0 sent and 0 failed with a key set means the send is still in "
                   "flight past flush()'s 5s window, not that nothing was queued. The "
                   "SDK prints a shutdown summary if anything ultimately failed; "
                   "otherwise the trace is in the dashboard."))
    if DAI_KEY:
        print(f"  app.decimal.ai/traces — filter agent_name = {T2['agent_name']}")
    else:
        print(wrap("enabled=False, so every decimalai.* call above was a no-op: nothing "
                   "was queued, nothing left this notebook, and nothing raised. Same "
                   "lines, same order, no second code path to rot."))

## 2.5 — What the world did with this skill

Your traces are yours. This is the public counterpart: the usage series behind the
chart on a registry skill's page, anonymous and unauthenticated like everything in
notebook 1.

It takes the skill's **UUID**, not its slug — the one place on this API where the two
are not interchangeable, and passing a slug returns a clean `404` that reads like *no
data* rather than *wrong identifier*. The cell demonstrates both so the failure is
recognisable when you hit it.

In [ ]:
# Wrapped because the registry API is the flakiest dependency in this notebook and a
# stack trace in the last cell of an otherwise clean run is a bad last impression.
skill, tot = {}, {}
try:
    skill = get(f"/registry/skills/{TARGET}")
    skill_id = skill["id"]
    try:
        by_slug = requests.get(f"{API}/registry/skills/{TARGET}/activations",
                               timeout=30)
        print(f"  by slug  → HTTP {by_slug.status_code}")
    except requests.RequestException as exc:
        print(f"  by slug  → {type(exc).__name__}")
    print(f"  by uuid  → HTTP 200   {skill_id}")
    print()

    act = get(f"/registry/skills/{skill_id}/activations", window_days=30,
              include="distinct_orgs,router_decisions,router_activated")
    tot = act.get("totals") or {}
    print(f"  window              {act.get('window_days')} days, "
          f"{len(act.get('series') or [])} buckets")
    print(f"  activations         {tot.get('activations', 0):,}")
    print(f"  routing decisions   {tot.get('router_decisions', 0):,}")
    print(f"  routed → activated  {tot.get('router_activated', 0):,}")
    print(f"  distinct orgs (max) {tot.get('distinct_orgs_max', 0):,}")
except Exception as exc:
    print(f"  the public usage series is unreachable right now "
          f"({type(exc).__name__}) — re-run this cell. The note below holds either "
          "way; it is about how to read the number, not about today's value.")
print()

# THE HONESTY NOTE, inline, because this is the number people misread.
print(wrap(
    "`total_activations_30d` — the field you meet on the skill page and in the browse "
    "API — is a LINEAGE total: the source skill PLUS every fork of it, summed. It is "
    "not 'this skill ran N times'. Comparing it against one skill_id's own traffic "
    "makes it look inflated by an order of magnitude; it cost us three phantom bug "
    "reports before we wrote that down. The series above is the same rollup, for the "
    "same reason: a fork is how you use a registry skill, so counting only the "
    "original would undercount it to zero."))

b = skill.get("benchmark_summary") or {}
lift = b.get("pass_rate_delta_pts") or 0
decisions, activated = tot.get("router_decisions", 0), tot.get("router_activated", 0)
if tot and decisions and not activated:
    print()
    print(wrap(
        f"Read the last two together. This skill was on the menu for {decisions:,} "
        f"routing decisions and was activated in {activated:,} of them, with a measured "
        f"{lift:+.2f} pts of lift sitting right there. Offered is not activated. That "
        "gap is the whole reason the routing_id you stamped in 2.4 exists — a "
        "benchmark can tell you a skill works, and only your traces can tell you "
        "anyone reached for it."))
elif tot and not decisions:
    print()
    print(wrap("No routing decisions recorded against this skill in the window, so "
               "there is nothing to say about offered-vs-activated yet. That is a "
               "real state and printing a zero beats printing a story."))

## 2.6 — Wiring it into your stack

Not executed — this notebook has no framework installed and running six lines of
LangChain to prove six lines of LangChain work is theatre. Copy what matches your
stack. **The caveats are the reason this section exists**; every one of them is a
thing the code does, or refuses to do, that the one-liner does not admit.

### LangChain / LangGraph

```python
import decimalai
from decimalai.langchain import install

decimalai.init(api_key="dai_sk_...")
install(agent_name="support-bot", enable_skill_loader=True)
# every chain / agent / llm call is now traced. Skill injection is NARROWER —
# see the bullet below before you assume your chain gets it.
```

- **Tracing and skill injection do not cover the same surface, and the gap is quiet.**
  Tracing is broad. Injection patches only `BaseChatModel.invoke` / `ainvoke`, and only
  rewrites an input that is a plain string or a message list. Measured:
  `llm.invoke("...")` injected, a message list injected, `.batch(...)` injected —
  but `prompt | llm` (LCEL hands `invoke` a `PromptValue`) is **not**, and
  `.stream(...)` is **not**. The adapter's input rewriter leaves an unrecognised shape
  untouched rather than guessing, which is the right call and also means an LCEL chain
  silently gets no skills. If you build with `prompt | llm`, inject the routed fragment
  into your own prompt template instead of relying on the patch.

- `enable_load_skill_tool=True` is accepted here and **does nothing**, by design. The
  adapter patches `BaseChatModel.invoke`, which is not a tool loop, so it cannot route
  a tool result back mid-turn. It logs a warning and stays on prompt injection. The
  live `load_skill` tool ships on `openai_agents` and `pydantic_ai` only.
- **`disk_sync` is load-bearing only when the skill loader is off.** Omitted, it
  resolves through `DecimalConfig.resolve_disk_sync(loader_active=...)`, which under
  the default `skill_authority="auto"` returns `not loader_active` — so the call
  above is already running with `disk_sync=False` without being told to, on purpose:
  with the router as the sole injection channel, mirroring the same skills to disk is
  how you get them injected twice. The pull lives inside `if disk_sync:`, so under the
  loader it never runs. The case to watch is the **loader off** one — a bare
  `install()`, or `decimalai.init(langchain=True)` as in 2.4 — which resolves to
  `True` and submits a background job that **pulls platform skills and writes
  `SKILL.md` files into your project**. Inside Claude Code or Cursor, which load those
  files themselves, that is the double injection. Pass `disk_sync=False` (or
  `init(skill_authority="router")`) *there* — passing it here changes nothing.

### OpenAI Agents SDK

```python
import decimalai
from decimalai.openai_agents import install

decimalai.init(api_key="dai_sk_...")
install(agent_name="support-bot", enable_skill_loader=True)
# Runner.run() / run_sync() are traced; `load_skill` is registered as a real tool
```

- This is the fullest path: traces *and* progressive disclosure. There is no
  `enable_load_skill_tool` flag because the tool is registered automatically whenever
  the skill loader is on. Turn it off with `init(load_skill_tool=False)` or
  `DECIMALAI_LOAD_SKILL_TOOL=0`.
- Pass `exclusive=True` only if you want to replace your existing trace processors;
  by default DecimalAI runs alongside OpenAI's own.

### Pydantic AI

```python
import decimalai
from decimalai.pydantic_ai import install

decimalai.init(api_key="dai_sk_...", anthropic=True)  # anthropic=True is the traces half
install(enable_skill_loader=True)
# every Agent() built after this gets a system_prompt function that calls the router,
# plus a live `load_skill` tool — same kill switch as the OpenAI Agents path
```

- **This adapter emits no traces at all.** It is a skill loader; tracing has to come
  from the provider SDK underneath, which is what the `anthropic=True` above turns on
  — swap in `openai=True` or `google=True` to match your model, or call
  `decimalai.providers.instrument()` to take whichever provider SDKs are importable.
  Each needs its OpenInference instrumentor installed, with the same silent-success
  trap as 2.4: missing instrumentor, pip hint in the log, nothing traced.
- What does **not** fill that gap is `decimalai.openai_agents.install()`. It registers
  a trace processor with the OpenAI *Agents SDK* — a framework a Pydantic AI agent
  never runs through — and raises `ImportError` unless `openai-agents` is installed.
  An adapter's `install()` instruments its own framework; it is not a generic tracing
  switch you can bolt onto another one.
- `decimalai.anthropic` is the same shape — skill loading only, no traces, pair it
  with `init(anthropic=True)` the same way — with one difference: its `install()`
  accepts `enable_load_skill_tool` and warns that it is dormant, where
  `pydantic_ai.install()` has no such parameter at all, because there the tool is
  live rather than dormant.

Longer, runnable versions of the first two:
[LangChain quickstart](../quickstart/quickstart_langchain.ipynb) ·
[OpenAI Agents quickstart](../quickstart/quickstart_openai_agents.ipynb)

## What you have now

A measured claim you re-ran yourself (notebook 1 and 2), a token bill you produced by
doing it the obvious way, a router that decides per request, and traces that record
which skill was offered, which was loaded, and which actually changed an answer.

The part that is still on you: **your traffic is the only evidence that matters about
your agent.** Our benchmark says this skill supplies knowledge one benchmark model
lacked, on cases we wrote. It does not say your users ask those questions, that your
model has the same gap, or that anyone will reach for it — which is exactly what the
offered-vs-activated number in 2.5 is for. Ship it, watch that number, and drop it if
it does not earn its place.

To leave no trace of this notebook in your workspace: the forks are at
app.decimal.ai/skills, and the traces are filed under the agent name pinned as
`agent_name` in the manifest.